In [ ]:
!pip install tensorflow opencv-python pandas pillow scikit-learn

In [ ]:
import pandas as pd
import numpy as np
import os
import cv2
from PIL import Image

import tensorflow as tf
from tensorflow.keras.applications import VGG16
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split
from tensorflow.keras.callbacks import EarlyStopping


In [ ]:
columns = [
    "Timestamp","CAN_ID","DLC",
    "D0","D1","D2","D3","D4","D5","D6","D7",
    "Class"
]

file_path = "DoS_dataset.csv"

if os.path.exists(file_path):
    df = pd.read_csv(file_path, names=columns)
    df = df.head(1000000)
    print("Dataset loaded. Size:", df.shape)
    display(df.head())
else:
    print("DoS_dataset.csv not found. Please run the upload cell above first.")

In [ ]:
if 'df' in locals():
    df = df.astype(str).apply(lambda x: x.str.strip())

    df['Class'] = df['Class'].replace({'R':0, 'T':1})
    df['Class'] = pd.to_numeric(df['Class'], errors='coerce')
    df = df.dropna(subset=['Class'])
    df['Class'] = df['Class'].astype(int)

    def safe_hex(x):
        try:
            return int(str(x), 16)
        except:
            return 0

    df['CAN_ID'] = df['CAN_ID'].apply(safe_hex)
    for col in ['D0','D1','D2','D3','D4','D5','D6','D7']:
        df[col] = df[col].apply(safe_hex)

    df['DLC'] = pd.to_numeric(df['DLC'], errors='coerce')
    df = df.fillna(0)

    feature_cols = ['CAN_ID','DLC','D0','D1','D2','D3','D4','D5','D6','D7']
    for col in feature_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    features = df[feature_cols].values
    labels = df['Class'].values
    print("Preprocessing complete. Features and labels ready.")
    print("Feature matrix shape:", features.shape)
else:
    print("Please load the dataset successfully first.")

In [ ]:
feature_min = features.min(axis=0)
feature_max = features.max(axis=0)
feature_range = np.where(feature_max - feature_min == 0, 1, feature_max - feature_min)
features_scaled = ((features - feature_min) / feature_range * 255).astype(np.uint8)

In [ ]:
WINDOW_SIZE = 25
FEATURES_PER_MSG = 10
SAMPLE_LEN = WINDOW_SIZE * FEATURES_PER_MSG

MAX_SAMPLES = 40000
num_windows = len(features_scaled) - WINDOW_SIZE + 1
num_samples = min(num_windows, MAX_SAMPLES)

X_list = []
y_list = []

for i in range(num_samples):
    window = features_scaled[i:i + WINDOW_SIZE].flatten()

    img = window.reshape(WINDOW_SIZE, FEATURES_PER_MSG)
    img = np.stack([img, img, img], axis=-1)
    img = cv2.resize(img, (224, 224))

    window_labels = labels[i:i + WINDOW_SIZE]
    label = 1 if window_labels.max() == 1 else 0

    X_list.append(img)
    y_list.append(label)

X = np.array(X_list)
y = np.array(y_list)

print("X shape:", X.shape)
print("y shape:", y.shape)

In [ ]:
X = X.astype(np.float32) / 255.0
print("X dtype:", X.dtype, "min:", X.min(), "max:", X.max())

In [ ]:
y_cat = to_categorical(y, num_classes=2)
print("y_cat shape after one-hot encoding:", y_cat.shape)

In [ ]:
X_train, X_temp, y_train, y_temp, y_train_int, y_temp_int = train_test_split(
    X, y_cat, y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

X_val, X_test, y_val, y_test, y_val_int, y_test_int = train_test_split(
    X_temp, y_temp, y_temp_int,
    test_size=0.50,
    random_state=42,
    stratify=y_temp_int
)

print("Train:", X_train.shape, "Val:", X_val.shape, "Test:", X_test.shape)

In [ ]:
base_model = VGG16(
    weights='imagenet',
    include_top=False,
    input_shape=(224,224,3)
)

for layer in base_model.layers:
    layer.trainable = False

In [ ]:
x = base_model.output
x = GlobalAveragePooling2D()(x)

x = Dense(256, activation='relu')(x)
x = Dropout(0.5)(x)

output = Dense(2, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=output)
model.summary()

In [ ]:
model.compile(
    optimizer=Adam(1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)
history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=20,
    batch_size=32,
    callbacks=[early_stop]
)

In [ ]:
test_loss, test_acc = model.evaluate(X_test, y_test)
print("Test Accuracy:", test_acc)
print("Test Loss:", test_loss)

In [ ]:
model.save("vgg16_can_dos_model.h5")
print("Model saved to vgg16_can_dos_model.h5")